# Dev: Generator Parameter System

Developer walkthrough of the `Generator` parameter system: initialization
parameters, fitted parameters, formatted summaries, and complete state
information. Not part of the published tutorials (the leading underscore
excludes it from the docs build).

## Create demonstration data and a generator

In [1]:
import numpy as np
import pandas as pd
from synhydro import ThomasFieringGenerator

np.random.seed(42)
dates = pd.date_range("2000-01-01", "2020-12-31", freq="MS")
flows = (
    1000
    + 500 * np.sin(np.arange(len(dates)) * 2 * np.pi / 12)
    + np.random.normal(0, 100, len(dates))
)
flows = np.maximum(flows, 50)  # Ensure positive
Q_monthly = pd.Series(flows, index=dates, name="site_A")

tf_gen = ThomasFieringGenerator(name="TF_Demo", debug=False)
print(tf_gen)

ThomasFieringGenerator(method='Thomas-Fiering AR(1)', distribution='Normal (after Stedinger transformation)')


## Initialization parameters (before fitting)

In [2]:
init_params = tf_gen.get_params()
for key, value in init_params.items():
    print(f"  {key}: {value}")

# Can also directly access the dataclass
tf_gen.init_params

  random_seed: None
  verbose: False
  debug: False
  method: Thomas-Fiering AR(1)
  distribution: Normal (after Stedinger transformation)
  transformation: SteddingerTransform
  by_month: True


GeneratorParams:
  Algorithm parameters:
    method: Thomas-Fiering AR(1)
    distribution: Normal (after Stedinger transformation)
  Transformation parameters:
    transformation: SteddingerTransform
    by_month: True

## Fit the generator

In [3]:
tf_gen.fit(Q_monthly)
print(tf_gen)

ThomasFieringGenerator(method='Thomas-Fiering AR(1)', distribution='Normal (after Stedinger transformation)')


## Fitted parameters (after fitting)

In [4]:
fitted_params = tf_gen.get_fitted_params()
for key, value in fitted_params.items():
    if isinstance(value, (int, str, tuple)):
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: <{type(value).__name__}>")

# Direct access to the fitted_params_ dataclass
tf_gen.fitted_params_

  means_: <dict>
  stds_: <dict>
  correlations_: <dict>
  distributions_: <dict>
  transformations_: <dict>
  n_parameters_: 36
  sample_size_: 252
  n_sites_: 1
  training_period_: ('2000-01-01', '2020-12-01')


FittedParams:
  n_parameters: 36
  sample_size: 252
  n_sites: 1
  training_period: 2000-01-01 to 2020-12-01
  means: fitted (Series of length 12)
  stds: fitted (Series of length 12)
  correlations: fitted ((12,))
  distributions: ['type', 'assumption']
  transformations: ['stedinger_transform']

## Comprehensive summary

In [5]:
print(tf_gen.summary())

                                TF_Demo Summary                                 

Model Information
--------------------------------------------------------------------------------
Generator Type:          ThomasFieringGenerator
Status:                  Fitted
Fitted:                  2026-07-30T10:59:05.106456
Number of Sites:         1
Sites:                   ['site_A']

Initialization Parameters
--------------------------------------------------------------------------------
  Algorithm parameters:
    method: Thomas-Fiering AR(1)
    distribution: Normal (after Stedinger transformation)
  Transformation parameters:
    transformation: SteddingerTransform
    by_month: True

Fitted Parameters
--------------------------------------------------------------------------------
  n_parameters: 36
  sample_size: 252
  n_sites: 1
  training_period: 2000-01-01 to 2020-12-01
  means: fitted (Series of length 12)
  stds: fitted (Series of length 12)
  correlations: fitted ((12,))
  distributi

## Complete state information

In [6]:
state_info = tf_gen.get_state_info()
print("State information keys:", list(state_info.keys()))
print(f"  - Class: {state_info['class']}")
print(f"  - Is fitted: {state_info['is_fitted']}")
print(f"  - Fit timestamp: {state_info['fit_timestamp']}")

State information keys: ['name', 'class', 'is_preprocessed', 'is_fitted', 'fit_timestamp', 'init_params', 'n_sites', 'sites', 'fitted_params']
  - Class: ThomasFieringGenerator
  - Is fitted: True
  - Fit timestamp: 2026-07-30T10:59:05.106456


## Generate synthetic data

In [7]:
Q_syn = tf_gen.generate(n_years=5, n_realizations=3)
first = Q_syn.data_by_realization[Q_syn.realization_ids[0]]
n_timesteps, n_sites = first.shape
print(f"Generated synthetic flows: {len(Q_syn.realization_ids)} realizations")
print(f"  {n_timesteps} timesteps x {n_sites} site(s) per realization")

Generated synthetic flows: 3 realizations
  60 timesteps x 1 site(s) per realization


## Access specific fitted parameters

In [8]:
print("Monthly means (normalized space):")
print(tf_gen.fitted_params_.means_)

print("Monthly standard deviations (normalized space):")
print(tf_gen.fitted_params_.stds_)

print("Monthly lag-1 correlations:")
print(tf_gen.fitted_params_.correlations_)

Monthly means (normalized space):
1     6.899213
2     7.100120
3     6.594420
4     7.318279
5     5.387881
6     6.321082
7     5.467804
8     6.598212
9     6.312687
10    6.212694
11    6.334763
12    6.652945
Name: site_A, dtype: float64
Monthly standard deviations (normalized space):
1     0.074488
2     0.076745
3     0.142018
4     0.055281
5     0.366793
6     0.206965
7     0.323171
8     0.147018
9     0.167056
10    0.164958
11    0.168828
12    0.146793
Name: site_A, dtype: float64
Monthly lag-1 correlations:
1    -0.038359
2    -0.105270
3    -0.299354
4    -0.003204
5    -0.301932
6     0.069948
7    -0.223972
8     0.177347
9    -0.259257
10    0.218570
11   -0.125482
12   -0.045809
Name: site_A, dtype: float64


## Key features

- `get_params()` - access initialization parameters
- `get_fitted_params()` - access learned parameters
- `summary()` - comprehensive formatted summary
- `get_state_info()` - complete state dictionary
- Direct access via the `init_params` and `fitted_params_` attributes